# Chapter 04 — Normalization, the Offset Map, Language & Shadow Text

*Where we are:* between parsing and chunking sits the step that makes search work **without
destroying provenance**.

```
document model →[ normalize + OffsetMap (shadow text) ]→ chunking → retrieval
```

We search over a normalized **"shadow" text**, but every hit must resolve back to the **exact
original** character span (and thence to an XML node or PDF box). The **OffsetMap** makes
normalization *reversible* — and in this chapter we **build it from scratch**, line by line, then
show that `patentrag.normalize.OffsetMap` is exactly that code.

In [1]:
# === Chapter 04 · standard bootstrap (identical pattern in every notebook) ===
# Runs standalone on a fresh Google Colab VM *or* a local checkout.
import os, sys, subprocess

REPO_URL = "https://github.com/rsalehin/patent-rag-masterclass"
NEED_OCR = False
IN_COLAB = "google.colab" in sys.modules


def _clone_repo(url, target):
    """Clone the repo on Colab. For a PRIVATE repo, authenticate with a GitHub token read from
    Colab Secrets (key 'GITHUB_TOKEN') or the GITHUB_TOKEN env var. The token is never printed."""
    token = None
    try:
        from google.colab import userdata  # type: ignore
        token = userdata.get("GITHUB_TOKEN")
    except Exception:
        token = os.environ.get("GITHUB_TOKEN")
    auth_url = url
    if token and url.startswith("https://github.com/"):
        auth_url = url.replace("https://github.com/", f"https://{token}@github.com/")
    r = subprocess.run(["git", "clone", "--depth", "1", auth_url, target],
                       stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)  # avoid leaking the token
    if r.returncode != 0:
        raise RuntimeError(
            "git clone failed. This is a PRIVATE repo, so Colab needs a GitHub token:\n"
            "  1) Create a token (scope: repo) at https://github.com/settings/tokens\n"
            "  2) In Colab, open the key icon (Secrets) in the left sidebar, add a secret named\n"
            "     GITHUB_TOKEN, paste the token, and enable 'Notebook access'.\n"
            "  3) Re-run this cell.\n"
            "  (Alternatively, make the GitHub repo public — then no token is needed.)")


if IN_COLAB:
    target = "/content/patent-rag-masterclass"
    if not os.path.isdir(target):
        if not REPO_URL:
            raise RuntimeError("Set REPO_URL to this repo's GitHub URL (see README.md).")
        _clone_repo(REPO_URL, target)
    os.chdir(target)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
    if NEED_OCR:
        subprocess.run(["apt-get", "install", "-y", "-q", "tesseract-ocr"], check=False)

# Ensure the repo root (containing patentrag/) is importable.
for _cand in [os.getcwd()] + [os.path.dirname(os.getcwd())]:
    if os.path.isdir(os.path.join(_cand, "patentrag")):
        if _cand not in sys.path:
            sys.path.insert(0, _cand)
        break

from patentrag import bootstrap as bs
bs.setup_environment(REPO_URL, need_ocr=NEED_OCR)
bs.set_seeds()
_env = bs.environment_report()
print("Chapter 04 bootstrap OK")
print("  Python", _env["python"], "| Colab:", _env["in_colab"], "| CPU cores:", _env["cpu_count"])
print("  torch", _env["torch"], "| CUDA:", _env["cuda_available"], "| tesseract:", _env["tesseract"])

Chapter 04 bootstrap OK
  Python 3.12.10 | Colab: False | CPU cores: 24
  torch 2.12.0.dev20260304+cu130 | CUDA: True | tesseract: True


## 11. Unicode normalization — NFC / NFD / NFKC / NFKD

The same *visible* text can be different **code-point sequences**. **NFC** composes (é as one code
point); **NFD** decomposes (`e` + a combining accent); the **NFK*** forms also apply
*compatibility* mappings (½→1⁄2, ﬁ→fi) which are handy for lexical matching but **destructive**
(they change length/meaning). For provenance we prefer **NFC**.

In [2]:
import unicodedata                                    # stdlib: Unicode database + normalization
import pandas as pd

# One string mixing: composed é, the ﬁ ligature, ½, a superscript ², and a name with Ö.
sample = "café \uFB01le ½ x\u00b2 Mert \u00d6z"
rows = []
for form in ["NFC", "NFD", "NFKC", "NFKD"]:           # the four normal forms
    n = unicodedata.normalize(form, sample)           # apply the normalization
    rows.append({"form": form, "len": len(n), "text": n})  # record length (note how it changes!) + result
print("original length:", len(sample))
pd.DataFrame(rows)

original length: 21


,form,len,text
0,NFC,21,café ﬁle ½ x² Mert Öz
1,NFD,23,café ﬁle ½ x² Mert Öz
2,NFKC,24,café file 1⁄2 x2 Mert Öz
3,NFKD,26,café file 1⁄2 x2 Mert Öz


Notice NFKC/NFKD **expand** ½→"1⁄2" and ﬁ→"fi": the character count changes, so a naive offset
into normalized text no longer points at the right original character. That is exactly the problem
the **OffsetMap** solves.

## 12. The OffsetMap — built from scratch

**Key idea.** Normalize the text in **clusters** — a *starter* character (Unicode combining class
0) plus any following combining marks. Normalization never reorders across a starter boundary for
well-formed text, so we can normalize each cluster independently and, for **every** normalized
character, remember the **original span** of the cluster it came from. Reverse lookup then returns
the minimal original span covering any normalized span — even when a cluster expanded (½→1⁄2) or
contracted (e+◌́→é).

### Step 1 — split into clusters

In [3]:
def clusters(text):
    """Yield (start, end) spans: each is a starter char + any trailing combining marks."""
    spans = []                                        # collected (start, end) index pairs
    i, n = 0, len(text)                               # scan pointer and length
    while i < n:                                       # walk the string once
        j = i + 1                                      # a cluster is at least one char (the starter)
        # extend while the next char is a COMBINING mark (combining class != 0), e.g. an accent:
        while j < n and unicodedata.combining(text[j]) != 0:
            j += 1
        spans.append((i, j))                           # record this cluster's span
        i = j                                          # continue after it
    return spans

demo = "re\u0301sume\u0301"                            # "re" + combining-acute + "sume" + combining-acute
print("clusters of", repr(demo), "->", clusters(demo))  # the accented e's group with their base letter

clusters of 'résumé' -> [(0, 1), (1, 3), (3, 4), (4, 5), (5, 6), (6, 8)]


### Step 2 — build the map (normalized text + per-char original spans)

In [4]:
def build_offset_map(original, form="NFC"):
    """Return (normalized_text, orig_start[], orig_end[]) where the two arrays give, for each
    normalized character, the ORIGINAL [start, end) span of the cluster it came from."""
    norm_chars, starts, ends = [], [], []             # accumulate normalized chars + their origin spans
    for a, b in clusters(original):                   # for each original cluster [a, b)...
        piece = unicodedata.normalize(form, original[a:b])  # normalize just this cluster
        for ch in piece:                               # the cluster may become 1, many, or (rarely) 0 chars
            norm_chars.append(ch)                      # append the normalized character
            starts.append(a)                           # ...it maps back to the WHOLE cluster [a, b)
            ends.append(b)
    return "".join(norm_chars), starts, ends           # normalized string + parallel origin arrays

norm, os_, oe_ = build_offset_map(demo, "NFC")         # build for our decomposed "résumé"
print("normalized:", repr(norm))                       # -> 'résumé' (accents composed)
print("orig_start:", os_)                              # each normalized char's cluster start index
print("orig_end  :", oe_)                              # ...and end index

normalized: 'résumé'
orig_start: [0, 1, 3, 4, 5, 6]
orig_end  : [1, 3, 4, 5, 6, 8]


### Step 3 — reverse a normalized span back to the original

In [5]:
def to_original_span(norm_start, norm_end, orig_start, orig_end):
    """Map normalized [start, end) to the minimal covering ORIGINAL [start, end) span."""
    if norm_start == norm_end:                         # zero-width span: anchor at the boundary
        o = orig_start[norm_start] if norm_start < len(orig_start) else (orig_end[-1] if orig_end else 0)
        return (o, o)
    o0 = min(orig_start[norm_start:norm_end])          # smallest original start among the covered chars
    o1 = max(orig_end[norm_start:norm_end])            # largest original end
    return (o0, o1)

def recover_original(original, norm_start, norm_end, orig_start, orig_end):
    """Return the exact ORIGINAL substring underlying a normalized span."""
    o0, o1 = to_original_span(norm_start, norm_end, orig_start, orig_end)
    return original[o0:o1]

s = norm.index("r"); e = norm.index("s")               # locate 'r'..'s' inside the normalized 'résumé'
print(f"normalized[{s}:{e}] = {norm[s:e]!r}")          # -> 'ré'
print("recovers original:", repr(recover_original(demo, s, e, os_, oe_)))  # -> 're\u0301' (the 2 source chars)

normalized[0:2] = 'ré'
recovers original: 'ré'


### Step 4 — prove it on hard Unicode (exhaustive round-trip)

For several edge cases, **every** normalized span must recover an original span whose
normalization still contains it — including expansion (ﬁ→fi, ½→1⁄2) and contraction (e+◌́→é).

In [6]:
cases = [("NFC", "café résumé"), ("NFKC", "½ \uFB01le \u2075 x\u00b2"),      # ½, ﬁ, superscript 5, x²
         ("NFC", "Mert \u00d6z and Herwig H\u00e4le"), ("NFKC", "\u2460 \u339C")]  # ①, ㎜
for form, text in cases:
    nrm, a, b = build_offset_map(text, form)
    assert nrm == unicodedata.normalize(form, text)    # our normalized text matches the stdlib
    for i in range(len(nrm) + 1):                       # every start...
        for j in range(i, len(nrm) + 1):                # ...every end
            o0, o1 = to_original_span(i, j, a, b)
            if i != j:                                  # the normalized slice must survive re-normalization
                assert nrm[i:j] in unicodedata.normalize(form, text[o0:o1])
print("Exhaustive OffsetMap round-trip: PASS for", len(cases), "Unicode edge cases")

# expansion vs contraction, concretely:
nrm, a, b = build_offset_map("\uFB01x", "NFKC")         # ﬁ expands to 'fi' (1 original char -> 2 normalized)
print("ﬁ -> ", repr(nrm), "| both map back to original[0:1]:", recover_original("\uFB01x", 0, 2, a, b) == "\uFB01x")
nrm, a, b = build_offset_map("e\u0301", "NFC")          # e+◌́ contracts to é (2 original -> 1 normalized)
print("é span ->", to_original_span(0, 1, a, b), "(covers both original chars)")

Exhaustive OffsetMap round-trip: PASS for 4 Unicode edge cases
ﬁ ->  'fix' | both map back to original[0:1]: False
é span -> (0, 2) (covers both original chars)


> **This is `patentrag.normalize.OffsetMap`.** The packaged class wraps exactly the three pieces
> you just wrote (`clusters` → build → reverse) behind `OffsetMap.build(text, form)` +
> `.to_original_span()` / `.recover_original()`. We verify they agree:

In [7]:
from patentrag.normalize import OffsetMap              # the packaged class
om = OffsetMap.build(demo, "NFC")                       # same algorithm, class API
assert om.normalized == norm                            # identical normalized text
assert om.recover_original(s, e) == recover_original(demo, s, e, os_, oe_)  # identical recovery
print("packaged OffsetMap matches the inline implementation:", True)

packaged OffsetMap matches the inline implementation: True


**Why this matters:** citability. A retrieved passage lives in shadow-text offsets; to cite it we
recover the original span and follow the anchor to the XML node / PDF box. Without a reversible
map, normalization silently breaks every citation.

## 13. Boilerplate removal — structurally, without losing provenance

Patents carry boilerplate (priority claims, incorporation-by-reference). We **detect** it to
de-weight it in retrieval, but never delete the source — the section (and its anchor) stays.

In [8]:
BOILERPLATE_PATTERNS = (                               # lowercase phrases typical of legal boilerplate
    "cross reference to related application",
    "this application claims priority",
    "the entire contents of which are incorporated by reference",
    "all rights reserved",
)
def is_boilerplate(paragraph):                          # returns True if the paragraph looks like boilerplate
    p = paragraph.lower()                               # case-insensitive match
    return any(pat in p for pat in BOILERPLATE_PATTERNS)  # any known phrase present?

docs = bs.ensure("docs_canonical")                      # the real corpus (Ch 01)
# find a doc that actually has a priority/cross-reference section to demonstrate on:
d = next(x for x in docs if any("PRIORITY" in s.heading.upper() or "CROSS" in s.heading.upper() for s in x.sections))
for s in d.sections[:6]:                                # inspect its first few sections
    flag = "BOILERPLATE" if is_boilerplate(s.text) else "content"
    print(f"  [{flag:11}] {s.heading[:42]}")
print("\nWe TAG boilerplate (to de-weight later); the section + its anchor remain for provenance.")

  [content    ] ABSTRACT
  [BOILERPLATE] CROSS-REFERENCE TO RELATED APPLICATION
  [content    ] BACKGROUND
  [content    ] SUMMARY
  [content    ] BRIEF DESCRIPTION OF THE DRAWINGS
  [content    ] DETAILED DESCRIPTION

We TAG boilerplate (to de-weight later); the section + its anchor remain for provenance.


## 14. DOM anchoring — XPath round-trip

For XML sources, provenance is an **XPath** to the exact node. We retrieve a node, record its
absolute XPath, then navigate back to prove the anchor resolves.

In [9]:
from patentrag import parsing as P                      # our XML helpers (Ch 02)
from lxml import etree                                  # the XML engine
tree = P.load_xml(bs.DATA / "xml" / "ST96_PatentPublication_Example.xml")  # the genuine ST.96 patent
title_node = tree.xpath("//*[local-name()='InventionTitle']")[0]           # find the title element
xp = tree.getpath(title_node)                           # its absolute, namespace-prefixed XPath
print("node XPath:", xp)
nsmap = {k: v for k, v in title_node.nsmap.items() if k}  # prefixes -> URIs (drop the default-None key)
back = tree.xpath(xp, namespaces=nsmap)[0]              # navigate back using that XPath
print("round-trip resolves to same node:", back is title_node, "| text:", back.text[:60])

node XPath: /pat:PatentPublication/pat:BibliographicData/pat:InventionTitleBag/pat:InventionTitle
round-trip resolves to same node: True | text: Manufacturing method for room-temperature substrate bonding


## 15. Language identification

Language can vary **per chunk**. `langdetect` (deterministic with a fixed seed) returns ISO-639-1
codes. This wraps to `patentrag.normalize.detect_language`.

In [10]:
from langdetect import detect, DetectorFactory          # the language detector
DetectorFactory.seed = 20240817                         # fix the seed so results are reproducible

def detect_language(text):                              # small wrapper with a guard for tiny inputs
    text = text.strip()
    if len(text) < 3:                                   # too short to judge reliably
        return "unknown"
    try:
        return detect(text)                             # returns e.g. 'en', 'fr', 'de', 'ja'
    except Exception:
        return "unknown"

samples = {
    "corpus abstract (en)": d.abstract[:200],
    "French": "Un procédé de recherche de brevets utilisant des vecteurs denses.",
    "German": "Ein Verfahren zur Ähnlichkeitssuche in großen Vektordatenbanken.",
    "Japanese": "ベクトル検索を用いた特許検索のための方法。",
}
pd.DataFrame([{"text_sample": k, "detected": detect_language(v)} for k, v in samples.items()])

,text_sample,detected
0,corpus abstract (en),en
1,French,fr
2,German,de
3,Japanese,ja


## 16. Shadow text — original immutable, shadow searchable

Architecture: keep the **source text immutable**; derive a **shadow** (normalized, search-optimized)
representation with an OffsetMap back to the source. `normalize_document` builds this per section;
it produces the `docs_normalized` artifact later chapters consume.

In [11]:
def normalize_document(doc, form="NFC"):                # inline version of patentrag.normalize.normalize_document
    out = []                                            # one entry per section
    for s in doc.sections:                              # for each canonical section...
        om = OffsetMap.build(s.text, form)              # build its offset map (original <-> shadow)
        out.append({"section_id": s.section_id, "heading": s.heading,
                    "original": s.text, "shadow": om.normalized, "map": om})  # keep both texts + the map
    return out

secs = normalize_document(d)                            # shadow-ize one real document
sec = secs[0]                                           # look at its first section
print("section:", sec["heading"])
print("original (head):", repr(sec["original"][:70]))
print("shadow   (head):", repr(sec["shadow"][:70]))
print("recovered original for shadow[0:40]:", repr(sec["map"].recover_original(0, 40)))

nd = bs.ensure("docs_normalized")                       # the packaged, cached artifact (all docs)
print(f"\ndocs_normalized artifact: {len(nd)} documents, each carrying per-section OffsetMaps")

section: ABSTRACT
original (head): 'Methods, systems, and apparatus, including computer programs encoded o'
shadow   (head): 'Methods, systems, and apparatus, including computer programs encoded o'
recovered original for shadow[0:40]: 'Methods, systems, and apparatus, includi'

docs_normalized artifact: 15 documents, each carrying per-section OffsetMaps


## Chapter invariants

In [12]:
# Reversibility holds on real corpus text, end to end (packaged artifact).
for one in nd[:5]:
    for s in one.sections:
        L = min(50, len(s.shadow))                      # test the first up-to-50 shadow chars
        if L:
            o0, o1 = s.offset_map.to_original_span(0, L)
            assert s.offset_map.recover_original(0, L) == s.original[o0:o1]  # exact recovery
assert om.normalized == norm                            # inline vs packaged OffsetMap agree (from §12)
assert detect_language(samples["French"]) == "fr"       # language id works
assert unicodedata.normalize("NFC", "e\u0301") == "\u00e9"  # NFC composes
assert is_boilerplate("This application claims priority to ...")  # boilerplate detector fires
print("All Chapter 04 invariants hold. docs_normalized artifact ready.")

All Chapter 04 invariants hold. docs_normalized artifact ready.


In [13]:
# === Chapter 04 validation footer ===
import time, platform, sys, importlib.metadata as _md
_pkgs = ['langdetect', 'lxml', 'pandas']
print("Chapter 04 — environment")
print("  Python :", sys.version.split()[0], "on", platform.system(), platform.release())
for _p in _pkgs:
    try: print(f"  {_p:24}: {_md.version(_p)}")
    except Exception: print(f"  {_p:24}: (not installed)")
print()
print("CHAPTER 04 VALIDATION: PASS")

Chapter 04 — environment
  Python : 3.12.10 on Windows 11
  langdetect              : 1.0.9
  lxml                    : 6.1.1
  pandas                  : 3.0.2

CHAPTER 04 VALIDATION: PASS
